In [1]:
import requests
import pandas as pd
import time
from pathlib import Path
import sys

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))
from src.polygonAPIkey import polygonAPIkey

API_KEY = polygonAPIkey
TICKER = "SPY"
YEAR = 2025
RAW_DATA_DIR = project_root / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import yfinance as yf
import pandas as pd

# Pull SPY price history for 2025 - trading days are just the days it traded
spy = yf.download("SPY", start="2025-01-01", end="2025-12-31", progress=False)
trading_days = spy.index.strftime("%Y-%m-%d").tolist()

print(f"Total trading days in 2025: {len(trading_days)}")
print(f"First: {trading_days[0]}  Last: {trading_days[-1]}")

Total trading days in 2025: 249
First: 2025-01-02  Last: 2025-12-30


In [4]:
def get_spot_price(date: str, api_key: str) -> float:
    url = f"https://api.polygon.io/v2/aggs/ticker/{TICKER}/range/1/day/{date}/{date}"
    r = requests.get(url, params={"adjusted": "true", "apiKey": api_key})
    r.raise_for_status()
    data = r.json()
    if data.get("resultsCount", 0) == 0:
        raise ValueError(f"No price data for {TICKER} on {date}")
    return data["results"][0]["c"]

In [5]:
def get_option_snapshot(date: str, spot: float, api_key: str) -> pd.DataFrame:
    url = f"https://api.polygon.io/v3/snapshot/options/{TICKER}"
    params = {
        "as_of": date,
        "limit": 250,
        "expiration_date_gte": (pd.Timestamp(date) + pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
        "expiration_date_lte": (pd.Timestamp(date) + pd.Timedelta(days=365)).strftime("%Y-%m-%d"),
        "strike_price_gte": round(spot * 0.80, 2),
        "strike_price_lte": round(spot * 1.20, 2),
        "apiKey": api_key
    }

    all_contracts = []
    page = 0

    while True:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()

        results = data.get("results", [])
        all_contracts.extend(results)
        page += 1

        next_url = data.get("next_url")
        if not next_url:
            break

        params = {"apiKey": api_key}
        url = next_url

    return pd.json_normalize(all_contracts)

In [6]:
skipped = []
completed = []

for date in trading_days:
    out_path = RAW_DATA_DIR / f"SPY_{date}.csv"

    # Skip if already pulled
    if out_path.exists():
        print(f"[SKIP] {date} already exists")
        completed.append(date)
        continue

    try:
        spot = get_spot_price(date, API_KEY)
        df = get_option_snapshot(date, spot, API_KEY)

        if df.empty:
            print(f"[WARN] {date} returned 0 contracts — skipping")
            skipped.append(date)
            continue

        # Save spot price as metadata in a separate column
        df["spot"] = spot
        df["trade_date"] = date
        df.to_csv(out_path, index=False)
        completed.append(date)
        print(f"[OK]   {date} — {len(df)} contracts saved")

    except Exception as e:
        print(f"[ERR]  {date} — {e}")
        skipped.append(date)

print(f"\nDone. {len(completed)} days completed, {len(skipped)} skipped/failed")
if skipped:
    print(f"Skipped dates: {skipped}")

[OK]   2025-01-02 — 13412 contracts saved
[OK]   2025-01-03 — 13412 contracts saved
[OK]   2025-01-06 — 13412 contracts saved
[OK]   2025-01-07 — 13412 contracts saved
[OK]   2025-01-08 — 13412 contracts saved
[OK]   2025-01-10 — 13412 contracts saved
[OK]   2025-01-13 — 13412 contracts saved
[OK]   2025-01-14 — 13412 contracts saved
[OK]   2025-01-15 — 13412 contracts saved
[OK]   2025-01-16 — 13412 contracts saved
[OK]   2025-01-17 — 13412 contracts saved
[OK]   2025-01-21 — 13412 contracts saved
[OK]   2025-01-22 — 13412 contracts saved
[OK]   2025-01-23 — 13412 contracts saved
[OK]   2025-01-24 — 13412 contracts saved
[OK]   2025-01-27 — 13412 contracts saved
[OK]   2025-01-28 — 13412 contracts saved
[OK]   2025-01-29 — 13412 contracts saved
[OK]   2025-01-30 — 13412 contracts saved
[OK]   2025-01-31 — 13412 contracts saved
[OK]   2025-02-03 — 13412 contracts saved
[OK]   2025-02-04 — 13412 contracts saved
[OK]   2025-02-05 — 13412 contracts saved
[OK]   2025-02-06 — 13412 contract